# Task 1 – Wildfire Data Cleaning

**Goal:** Build a clean zip-year level table from the raw wildfire data, then create a final
per-zip feature table that can feed directly into the QML model.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/wildfire_data.csv", low_memory=False)

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()

Shape: (125476, 30)
['OBJECTID', 'Year', 'AGENCY', 'UNIT_ID', 'FIRE_NAME', 'INC_NUM', 'ALARM_DATE', 'CONT_DATE', 'CAUSE', 'C_METHOD', 'OBJECTIVE', 'GIS_ACRES', 'COMMENTS', 'COMPLEX_NA', 'FIRE_NUM', 'COMPLEX_ID', 'DECADES', 'Shape__Are', 'Shape__Len', 'latitude', 'longitude', 'zip', 'Alarm_Date2', 'year_month', 'AGENCY_ID', 'FIRE_NAME_ID', 'avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm', 'station']


,OBJECTID,Year,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,ALARM_DATE,CONT_DATE,CAUSE,C_METHOD,...,longitude,zip,Alarm_Date2,year_month,AGENCY_ID,FIRE_NAME_ID,avg_tmax_c,avg_tmin_c,tot_prcp_mm,station
0,1653.0,2019.0,CDF,MEU,GRADE,00008307,7/16/17,7/21/17,10.0,7.0,...,-123.266813,95470.0,2017-07-16,2017-07,0.0,92.0,NaN,NaN,NaN,NaN
1,1982.0,2018.0,CCO,VNC,SOUTH,00113365,12/21/17,12/21/17,14.0,6.0,...,-119.051707,93060.0,2017-12-21,2017-12,2.0,102.0,NaN,NaN,NaN,NaN
2,1980.0,2018.0,CCO,VNC,BALCOM,00113691,12/22/17,12/22/17,14.0,6.0,...,-118.977639,93066.0,2017-12-22,2017-12,2.0,1400.0,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,90001.0,NaN,2018-01,NaN,NaN,21.748387,11.432258,35.5,72295.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,90002.0,NaN,2018-01,NaN,NaN,21.748387,11.432258,35.5,72295.0


## Split into Fire Incidents vs Weather Rows

The raw file has two fundamentally different row types mixed together:
- **Fire rows**: `OBJECTID` is not null — one row per fire incident (row 0,1,2)
- **Weather rows**: `OBJECTID` is null — one row per zip × month, with temperature/precipitation data (row 3,4)

In [2]:
fire_raw = df[df['OBJECTID'].notna()]
weather_raw = df[df['OBJECTID'].isna()]

print("Fire rows:", len(fire_raw))
print("Weather rows:", len(weather_raw))

Fire rows: 2218
Weather rows: 123258


## Fire Data
- Data cleaning for fire data
- There was 2017 data so I changed the range from 2018 to 2023.

In [3]:
# this tells me that there is 2017 data, and we don't need this data.
fire_raw = fire_raw.copy()
fire_raw['alarm_year'] = pd.to_datetime(fire_raw['Alarm_Date2'], errors='coerce').dt.year
print(fire_raw['alarm_year'].value_counts().sort_index())
print(f"Before year filter (2017-2023): {len(fire_raw):,} incidents")

# so, i decided to filter the data from 2018 to 2023.
fire = fire_raw[fire_raw['alarm_year'].between(2018, 2023)].copy()
print(f"After year filter (2018-2023): {len(fire):,} incidents")

# i got -6 instead of -3 because...
print("\n")
print(f"There are {fire_raw['alarm_year'].isna().sum()} data are has NaN")
print(fire_raw[fire_raw['alarm_year'].isna()][['FIRE_NAME', 'ALARM_DATE', 'Alarm_Date2', 'alarm_year']])

alarm_year
2017.0      3
2018.0    413
2019.0    313
2020.0    504
2021.0    389
2022.0    306
2023.0    284
Name: count, dtype: int64
Before year filter (2017-2023): 2,218 incidents
After year filter (2018-2023): 2,209 incidents


There are 6 data are has NaN
          FIRE_NAME ALARM_DATE Alarm_Date2  alarm_year
125470         STAR        NaN         NaN         NaN
125471  SKYLINE LRA        NaN         NaN         NaN
125472     TAMARACK        NaN         NaN         NaN
125473       BORDER        NaN         NaN         NaN
125474         YORK        NaN         NaN         NaN
125475        KELTY        NaN         NaN         NaN


## Definition of Wildfire
Wildfire: To reduce complexity, a ‘wildfire’ will be defined by if it burns in a wildland setting and is unplanned and uncontrolled. This is from the challenge document.

- Based on external wildfire coding references and the challenge definition, I filtered to OBJECTIVE == 1.0 to keep suppression fires, which best match unplanned and uncontrolled wildfires. I excluded OBJECTIVE == 2.0 prescribed burns because they are planned controlled fires.

In [4]:
#  I ran print(fire['OBJECTIVE'].value_counts()) and I got
#  OBJECTIVE 1.0    2185 2.0      18
#  So I added dropna = False to count the number of NaN data
print(fire['OBJECTIVE'].value_counts(dropna=False))

fire = fire[fire['OBJECTIVE'] == 1.0].copy()
print(f"After OBJECTIVE filter: {len(fire):,} incidents")

print(fire['zip'].dtype)
print(fire['zip'].isna().sum())

fire = fire.dropna(subset=['zip']).copy()
print(f"After zip filter: {len(fire):,} incidents")
# changed zip type to integer
fire['zip'] = fire['zip'].astype(int)
print(fire['zip'].dtype)
print(fire['zip'].head())

OBJECTIVE
1.0    2185
2.0      18
NaN       6
Name: count, dtype: int64
After OBJECTIVE filter: 2,185 incidents
float64
321
After zip filter: 1,864 incidents
int64
133    90265
388    91360
946    92630
955    92653
969    92675
Name: zip, dtype: int64


## Cause?

In [5]:
print(fire['CAUSE'].value_counts(dropna=False))


CAUSE
14.0    702
2.0     238
9.0     180
1.0     177
10.0    169
11.0    117
7.0     101
5.0      85
4.0      32
8.0      21
18.0     14
15.0     12
3.0      11
6.0       3
16.0      2
Name: count, dtype: int64


In [6]:
fire_agg = fire.groupby(['zip', 'alarm_year']).agg(
    fire_count=('OBJECTID', 'count'),
    total_acres=('GIS_ACRES', 'sum')
).reset_index()

print(fire_agg.shape)
print(fire_agg.head())

all_zips = weather_raw['zip'].dropna().unique()
print(f"Total unique zips: {len(all_zips)}")

all_years = list(range(2018, 2024))

grid = pd.MultiIndex.from_product([all_zips, all_years], names=['zip', 'alarm_year'])
grid = pd.DataFrame(index=grid).reset_index()
grid['zip'] = grid['zip'].astype(int)

grid = grid.merge(fire_agg, on=['zip', 'alarm_year'], how='left')
grid['fire_count'] = grid['fire_count'].fillna(0)
grid['total_acres'] = grid['total_acres'].fillna(0)

print(grid.shape)
print(grid.head(10))

(1168, 4)
     zip  alarm_year  fire_count  total_acres
0  85364      2020.0           1    17.045660
1  89029      2020.0           1    42.103680
2  89439      2019.0           1     5.259314
3  89439      2023.0           1    52.934110
4  89508      2019.0           1  2438.542000
Total unique zips: 2593
(15558, 4)
     zip  alarm_year  fire_count  total_acres
0  90001        2018         0.0          0.0
1  90001        2019         0.0          0.0
2  90001        2020         0.0          0.0
3  90001        2021         0.0          0.0
4  90001        2022         0.0          0.0
5  90001        2023         0.0          0.0
6  90002        2018         0.0          0.0
7  90002        2019         0.0          0.0
8  90002        2020         0.0          0.0
9  90002        2021         0.0          0.0


## Fire Occurred

In [7]:
grid['fire_occurred'] = (grid['fire_count'] > 0).astype(int)

print(grid['fire_occurred'].value_counts())

fire_zips = set(fire['zip'].unique())
weather_zips = set(grid['zip'].unique())


fire_occurred
0    14399
1     1159
Name: count, dtype: int64


In [8]:
weather_raw['year'] = weather_raw['year_month'].str[:4].astype(int)
weather_raw['zip'] = weather_raw['zip'].dropna().astype(int)

weather_agg = weather_raw.groupby(['zip', 'year']).agg(
    avg_tmax_c=('avg_tmax_c', 'mean'),
    avg_tmin_c=('avg_tmin_c', 'mean'),
    tot_prcp_mm=('tot_prcp_mm', 'sum')
).reset_index()

print(weather_agg.shape)
print(weather_agg.head())




(10372, 5)
     zip  year  avg_tmax_c  avg_tmin_c  tot_prcp_mm
0  90001  2018   22.319292   14.603904        198.1
1  90001  2019   21.559567   13.958907        475.7
2  90001  2020   22.298544   14.032893        229.6
3  90001  2021   20.878253   13.234925        307.3
4  90002  2018   22.319292   14.603904        198.1


/var/folders/8g/24hb2lpj4wx0b6njqx5m9m6c0000gn/T/ipykernel_6217/2298692962.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_raw['year'] = weather_raw['year_month'].str[:4].astype(int)
/var/folders/8g/24hb2lpj4wx0b6njqx5m9m6c0000gn/T/ipykernel_6217/2298692962.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_raw['zip'] = weather_raw['zip'].dropna().astype(int)


In [9]:
grid = grid.merge(weather_agg, 
                  left_on=['zip', 'alarm_year'], 
                  right_on=['zip', 'year'], 
                  how='left')

grid = grid.drop(columns=['year'])

print(grid.shape)
print(grid.head())



(15558, 8)
     zip  alarm_year  fire_count  total_acres  fire_occurred  avg_tmax_c  \
0  90001        2018         0.0          0.0              0   22.319292   
1  90001        2019         0.0          0.0              0   21.559567   
2  90001        2020         0.0          0.0              0   22.298544   
3  90001        2021         0.0          0.0              0   20.878253   
4  90001        2022         0.0          0.0              0         NaN   

   avg_tmin_c  tot_prcp_mm  
0   14.603904        198.1  
1   13.958907        475.7  
2   14.032893        229.6  
3   13.234925        307.3  
4         NaN          NaN  


In [10]:
for col in ['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']:
    grid[col] = grid[col].fillna(grid[col].median())

print(grid.isnull().sum())

zip              0
alarm_year       0
fire_count       0
total_acres      0
fire_occurred    0
avg_tmax_c       0
avg_tmin_c       0
tot_prcp_mm      0
dtype: int64


In [11]:
grid.to_csv("../data/processed/wildfire_cleaned.csv", index=False)
print("saved!")
print(grid.shape)

saved!
(15558, 8)
